# ASVspoof5 Train-Only Gender-Specific Sampling (A01-A04 + bonafide)

Builds two balanced subsets from `ASVspoof5.train.tsv`:
- `male`: 50 speakers
- `female`: 50 speakers

For each gender subset:
- classes: `bonafide`, `A01`, `A02`, `A03`, `A04`
- exact class size: `2000` each
- split: `70/30` speaker-disjoint (`train` / `test`)
- per split class targets: `1400 train`, `600 test`

Exports per gender:
- `selected_speakers.csv`
- `selected_utterances_plan.csv`
- `selection_audit.csv`
- `plan_summary.json`


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd


In [2]:
# ===== Config =====
PROJECT_ROOT = Path('/home/SpeakerRec/BioVoice')
PROTOCOL_PATH = PROJECT_ROOT / 'data' / 'datasets' / 'ASVspoof5_tars' / 'ASVspoof5_protocols' / 'ASVspoof5.train.tsv'

OUT_BASE = PROJECT_ROOT / 'ASVspoof5_protocols' / 'gender_50spk_2000perclass_A01A04_outputs'
OUT_BASE.mkdir(parents=True, exist_ok=True)

SEED = 42
SPEAKERS_PER_GENDER = 50
TRAIN_SPK = 35
TEST_SPK = 15

TARGET_CLASSES = ['bonafide', 'A01', 'A02', 'A03', 'A04']
TARGET_PER_CLASS_TOTAL = 2000
TARGET_PER_CLASS_TRAIN = 1400
TARGET_PER_CLASS_TEST = 600

MAX_SELECTION_TRIES = 5000

print('PROTOCOL_PATH =', PROTOCOL_PATH, '| exists =', PROTOCOL_PATH.exists())
print('OUT_BASE =', OUT_BASE)


PROTOCOL_PATH = /home/SpeakerRec/BioVoice/data/datasets/ASVspoof5_tars/ASVspoof5_protocols/ASVspoof5.train.tsv | exists = True
OUT_BASE = /home/SpeakerRec/BioVoice/ASVspoof5_protocols/gender_50spk_2000perclass_A01A04_outputs


In [3]:
# ===== Load protocol =====
assert PROTOCOL_PATH.exists(), f'Missing protocol: {PROTOCOL_PATH}'

cols = ['speaker_id','utt_id','gender','c4','c5','c6','codec_id','system_id','label','unused']
df = pd.read_csv(PROTOCOL_PATH, sep=r'\s+', header=None, names=cols, engine='python')

# Normalize class label for sampling
sample_class = np.where(df['label'].eq('bonafide'), 'bonafide', df['system_id'].astype(str))
df['sample_class'] = sample_class

# Keep only required classes
keep = set(TARGET_CLASSES)
df = df[df['sample_class'].isin(keep)].copy().reset_index(drop=True)

print('Rows after class filter:', len(df))
print('Speakers:', df['speaker_id'].nunique())
print('Gender counts:', df['gender'].value_counts().to_dict())
print('Class counts:')
print(df['sample_class'].value_counts().sort_index())


Rows after class filter: 100577
Speakers: 400
Gender counts: {'M': 50836, 'F': 49741}
Class counts:
A01         20445
A02         20445
A03         20445
A04         20445
bonafide    18797
Name: sample_class, dtype: int64


In [4]:
def speaker_class_table(df_g: pd.DataFrame) -> pd.DataFrame:
    piv = df_g.groupby(['speaker_id','sample_class']).size().unstack(fill_value=0)
    for c in TARGET_CLASSES:
        if c not in piv.columns:
            piv[c] = 0
    piv = piv[TARGET_CLASSES]
    gender = df_g.groupby('speaker_id')['gender'].first().rename('gender')
    return piv.join(gender, how='left').reset_index()


def select_speakers_and_split(df_g: pd.DataFrame, gender_name: str, seed: int):
    tab = speaker_class_table(df_g)

    # Strict: each chosen speaker must have at least 1 sample in every class.
    eligible = tab[(tab[TARGET_CLASSES] >= 1).all(axis=1)].copy().reset_index(drop=True)
    assert len(eligible) >= SPEAKERS_PER_GENDER, (
        f'Not enough eligible {gender_name} speakers with all classes present: {len(eligible)}'
    )

    rng = np.random.default_rng(seed)

    for t in range(MAX_SELECTION_TRIES):
        sel_idx = rng.choice(len(eligible), size=SPEAKERS_PER_GENDER, replace=False)
        sel = eligible.iloc[sel_idx].copy().reset_index(drop=True)

        # random split speakers 35/15
        perm = rng.permutation(len(sel))
        tr = sel.iloc[perm[:TRAIN_SPK]].copy()
        te = sel.iloc[perm[TRAIN_SPK:TRAIN_SPK+TEST_SPK]].copy()

        # availability checks per split per class
        tr_av = tr[TARGET_CLASSES].sum()
        te_av = te[TARGET_CLASSES].sum()

        ok_train = all(int(tr_av[c]) >= TARGET_PER_CLASS_TRAIN for c in TARGET_CLASSES)
        ok_test = all(int(te_av[c]) >= TARGET_PER_CLASS_TEST for c in TARGET_CLASSES)
        if ok_train and ok_test:
            tr['split'] = 'train'
            te['split'] = 'test'
            out = pd.concat([tr, te], ignore_index=True)
            out['gender_set'] = gender_name
            out = out[['gender_set','split','speaker_id','gender'] + TARGET_CLASSES]
            return out

    raise RuntimeError(
        f'Could not find feasible {gender_name} speaker selection after {MAX_SELECTION_TRIES} tries '
        f'for required split quotas train={TARGET_PER_CLASS_TRAIN}, test={TARGET_PER_CLASS_TEST}.'
    )


def sample_manifest_for_gender(df_g: pd.DataFrame, speaker_plan: pd.DataFrame, seed: int):
    rng = np.random.default_rng(seed)
    selected_rows = []
    audit_rows = []

    for split_name, target_n in [('train', TARGET_PER_CLASS_TRAIN), ('test', TARGET_PER_CLASS_TEST)]:
        spk_set = set(speaker_plan.loc[speaker_plan['split'].eq(split_name), 'speaker_id'].astype(str).tolist())
        split_df = df_g[df_g['speaker_id'].astype(str).isin(spk_set)].copy()

        for cls in TARGET_CLASSES:
            pool = split_df[split_df['sample_class'].eq(cls)].copy()
            avail = len(pool)
            if avail < target_n:
                raise RuntimeError(f'Insufficient pool for {cls} in {split_name}: avail={avail}, need={target_n}')
            pick_idx = rng.choice(pool.index.to_numpy(), size=target_n, replace=False)
            pick = pool.loc[pick_idx].copy()
            pick['split'] = split_name
            pick['target_class'] = cls
            selected_rows.append(pick)
            audit_rows.append({
                'split': split_name,
                'target_class': cls,
                'target_n': int(target_n),
                'selected_n': int(len(pick)),
                'availability_n': int(avail),
            })

    manifest = pd.concat(selected_rows, ignore_index=True)

    # Ensure no duplicate utt across class/split selections
    assert manifest['utt_id'].nunique() == len(manifest), 'Duplicate utt_id selected'

    # Keep consistent columns
    manifest = manifest[[
        'split','speaker_id','utt_id','gender','label','system_id','sample_class','codec_id','target_class'
    ]].sort_values(['split','target_class','speaker_id','utt_id']).reset_index(drop=True)

    audit_df = pd.DataFrame(audit_rows).sort_values(['split','target_class']).reset_index(drop=True)
    return manifest, audit_df


In [5]:
# ===== Build male/female datasets =====
results = {}

for gender_name, gval, seed_off in [('male','M',100), ('female','F',200)]:
    print(f'\n=== Building {gender_name} set ===')
    df_g = df[df['gender'].eq(gval)].copy().reset_index(drop=True)
    print('rows=', len(df_g), 'speakers=', df_g['speaker_id'].nunique())

    speaker_plan = select_speakers_and_split(df_g, gender_name, seed=SEED + seed_off)
    manifest, audit_df = sample_manifest_for_gender(df_g, speaker_plan, seed=SEED + seed_off + 1)

    # Integrity checks
    assert speaker_plan['speaker_id'].nunique() == SPEAKERS_PER_GENDER
    assert set(speaker_plan['split'].unique()) == {'train','test'}
    assert len(set(speaker_plan[speaker_plan['split']=='train']['speaker_id']).intersection(
        set(speaker_plan[speaker_plan['split']=='test']['speaker_id'])
    )) == 0, 'speaker leakage train/test'

    class_totals = manifest['target_class'].value_counts().to_dict()
    for c in TARGET_CLASSES:
        assert int(class_totals.get(c, 0)) == TARGET_PER_CLASS_TOTAL, f'{gender_name} wrong total for {c}'

    split_class = manifest.groupby(['split','target_class']).size().unstack(fill_value=0)
    for c in TARGET_CLASSES:
        assert int(split_class.loc['train', c]) == TARGET_PER_CLASS_TRAIN
        assert int(split_class.loc['test', c]) == TARGET_PER_CLASS_TEST

    out_dir = OUT_BASE / gender_name
    out_dir.mkdir(parents=True, exist_ok=True)

    speakers_csv = out_dir / 'selected_speakers.csv'
    manifest_csv = out_dir / 'selected_utterances_plan.csv'
    audit_csv = out_dir / 'selection_audit.csv'
    summary_json = out_dir / 'plan_summary.json'

    speaker_plan.to_csv(speakers_csv, index=False)
    manifest.to_csv(manifest_csv, index=False)
    audit_df.to_csv(audit_csv, index=False)

    summary = {
        'gender_set': gender_name,
        'seed': int(SEED + seed_off),
        'protocol': str(PROTOCOL_PATH),
        'speaker_plan': {
            'total_speakers': int(SPEAKERS_PER_GENDER),
            'train_speakers': int(TRAIN_SPK),
            'test_speakers': int(TEST_SPK),
        },
        'class_targets': {
            'classes': TARGET_CLASSES,
            'total_per_class': int(TARGET_PER_CLASS_TOTAL),
            'train_per_class': int(TARGET_PER_CLASS_TRAIN),
            'test_per_class': int(TARGET_PER_CLASS_TEST),
        },
        'counts': {
            'manifest_rows': int(len(manifest)),
            'unique_utt_id': int(manifest['utt_id'].nunique()),
            'class_totals': {k:int(v) for k,v in manifest['target_class'].value_counts().sort_index().to_dict().items()},
            'split_class_totals': {
                s: {k:int(v) for k,v in g['target_class'].value_counts().sort_index().to_dict().items()}
                for s, g in manifest.groupby('split')
            }
        }
    }
    summary_json.write_text(json.dumps(summary, indent=2), encoding='utf-8')

    results[gender_name] = {
        'out_dir': str(out_dir),
        'rows': int(len(manifest)),
        'speakers': int(speaker_plan['speaker_id'].nunique()),
        'class_totals': {k:int(v) for k,v in manifest['target_class'].value_counts().sort_index().to_dict().items()},
    }

    print('Saved ->', out_dir)
    print('class totals:', results[gender_name]['class_totals'])

print('\nDone. Summary:')
print(json.dumps(results, indent=2))



=== Building male set ===
rows= 50836 speakers= 204
Saved -> /home/SpeakerRec/BioVoice/ASVspoof5_protocols/gender_50spk_2000perclass_A01A04_outputs/male
class totals: {'A01': 2000, 'A02': 2000, 'A03': 2000, 'A04': 2000, 'bonafide': 2000}

=== Building female set ===
rows= 49741 speakers= 196
Saved -> /home/SpeakerRec/BioVoice/ASVspoof5_protocols/gender_50spk_2000perclass_A01A04_outputs/female
class totals: {'A01': 2000, 'A02': 2000, 'A03': 2000, 'A04': 2000, 'bonafide': 2000}

Done. Summary:
{
  "male": {
    "out_dir": "/home/SpeakerRec/BioVoice/ASVspoof5_protocols/gender_50spk_2000perclass_A01A04_outputs/male",
    "rows": 10000,
    "speakers": 50,
    "class_totals": {
      "A01": 2000,
      "A02": 2000,
      "A03": 2000,
      "A04": 2000,
      "bonafide": 2000
    }
  },
  "female": {
    "out_dir": "/home/SpeakerRec/BioVoice/ASVspoof5_protocols/gender_50spk_2000perclass_A01A04_outputs/female",
    "rows": 10000,
    "speakers": 50,
    "class_totals": {
      "A01": 2000,
 